In [39]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder ,OneHotEncoder
import pickle

In [40]:
data = pd.read_csv('./Dataset/Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


### Preprocessing the data

In [41]:
data = data.drop(['RowNumber','CustomerId','Surname'],axis=1)

#### Encode categorical Variable

In [42]:
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])


#### One Hot Encode Geography

In [43]:
onehot_encoder_geography = OneHotEncoder()
geo_encoded = onehot_encoder_geography.fit_transform(data[['Geography']])
geo_encoded_df = pd.DataFrame(geo_encoded.toarray(), columns=onehot_encoder_geography.get_feature_names_out(['Geography']))

Combining The columns with original

In [44]:
data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [45]:
#spilting the data into train and test
X = data.drop('EstimatedSalary', axis=1)
y = data['EstimatedSalary']

In [46]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [47]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [48]:
#Save the preprocessor objects
with open('label_encoder_gender.pkl', 'wb') as f:
    pickle.dump(label_encoder_gender, f)

with open('onehot_encoder_geography.pkl', 'wb') as f:
    pickle.dump(onehot_encoder_geography, f)    

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

### Train ANN Regression Problem Statement

In [49]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [50]:
#Building the model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1)  # Output layer for regression
])

##Compile the model
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mean_absolute_error'])
model.summary()


Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_3 (Dense)             (None, 64)                832       
                                                                 
 dense_4 (Dense)             (None, 32)                2080      
                                                                 
 dense_5 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [51]:
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

log_dir = "Regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [52]:
## Set early stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights = True)

In [53]:
##Train the model
history = model.fit(X_train_scaled, y_train, epochs=100, batch_size=32, validation_split=0.2, callbacks=[early_stopping, tensorboard_callback])

Epoch 1/100
200/200 [==============================] - 1s 4ms/step - loss: 13310886912.0000 - mean_absolute_error: 99991.5781 - val_loss: 13706214400.0000 - val_mean_absolute_error: 102078.8906
Epoch 2/100
200/200 [==============================] - 1s 3ms/step - loss: 13268410368.0000 - mean_absolute_error: 99779.2031 - val_loss: 13616062464.0000 - val_mean_absolute_error: 101637.5469
Epoch 3/100
200/200 [==============================] - 1s 3ms/step - loss: 13106580480.0000 - mean_absolute_error: 98972.5312 - val_loss: 13366258688.0000 - val_mean_absolute_error: 100419.6328
Epoch 4/100
200/200 [==============================] - 1s 3ms/step - loss: 12758453248.0000 - mean_absolute_error: 97238.0938 - val_loss: 12903359488.0000 - val_mean_absolute_error: 98159.5312
Epoch 5/100
200/200 [==============================] - 1s 3ms/step - loss: 12186471424.0000 - mean_absolute_error: 94400.6875 - val_loss: 12203955200.0000 - val_mean_absolute_error: 94744.3828
Epoch 6/100
200/200 [===========

In [54]:
%load_ext tensorboard
%tensorboard --logdir logs/fit

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 17728), started 7:41:36 ago. (Use '!kill 17728' to kill it.)

In [55]:
## Evaluate the model
test_loss, test_mae = model.evaluate(X_test_scaled, y_test)
print(f'Test Loss: {test_loss}, Test MAE: {test_mae}')

63/63 [==============================] - 0s 2ms/step - loss: 3357463296.0000 - mean_absolute_error: 50069.6250
Test Loss: 3357463296.0, Test MAE: 50069.625


In [56]:
model.save('salary_regression_model.h5')

c:\Users\samya\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
